# Monocular Depth Estimation — AIGOAT 1.0, Task 2

**Team:** PPP: PeniParkersPrime &nbsp;·&nbsp; **Final rank: #3** &nbsp;·&nbsp; **Score: 1.47**

This notebook trains a compact neural network that predicts a per-pixel **depth map** from a single RGB image — i.e. how far each pixel is from the camera — without any stereo pair, LiDAR, or camera metadata. It was our submission to Task 2 of the AIGOAT 1.0 hackathon.

The competition does not score accuracy alone. The leaderboard uses a **composite score** that multiplies three factors, so a winning model has to be accurate *and* small *and* fast:

> `Score = Accuracy(RMSE) × SizeScore(MB) × SpeedScore(seconds)`

That constraint shaped every design decision below: we deliberately chose a small mobile-grade backbone, exported the model in half precision to keep it under 4 MB, and folded preprocessing into the network so inference stays fast.

---

## Results we achieved

| Metric | Value |
|---|---|
| Validation RMSE | **0.1077** (limit was 0.16) |
| Model size (ONNX, FP16) | **3.71 MB** |
| Parameters | 1.93 M |
| Inference (batch of 8, GPU) | ~1.37 s on eval server |
| **Composite score** | **1.47 → Rank #3** |

## How the model works — the key ideas

1. **MobileNetV2 encoder + depthwise-separable decoder.** A lightweight U-Net-style architecture. Using a mobile backbone keeps the parameter count (and therefore file size) tiny.
2. **Knowledge distillation from MiDaS.** We run the pretrained [MiDaS](https://github.com/isl-org/MiDaS) depth model over the training set once and use its predictions as an extra "teacher" target. Our small model learns to imitate MiDaS's scene structure on top of the ground-truth labels (the teacher aligned with the labels at Pearson r = 0.985).
3. **FP16 ONNX export.** Weights are stored in half precision, roughly halving the file size (7.7 MB → 3.71 MB) for a big boost to the size score.
4. **Folded ImageNet normalization.** Instead of normalizing inputs at runtime, we bake the ImageNet mean/std directly into the weights of the first convolution. The model accepts raw `[0,1]` RGB — no preprocessing step, which is also what the eval server requires.
5. **EMA (exponential moving average) of weights** for smoother, better-generalizing final weights.
6. **A combined loss** — L1 + gradient matching + SSIM + a scheduled scale-invariant term — to capture both per-pixel accuracy and overall scene structure.

> **Note on reproducibility.** This notebook was run on Kaggle with a Tesla T4 GPU against the competition dataset (paths under `/kaggle/input`). The cell outputs below are preserved from that original run. It is **not** re-runnable here as-is — it needs the Kaggle dataset and GPU environment.

The single code cell below is self-contained: it handles dependencies, data loading, the teacher pass, model definition, training, ONNX export, and benchmarking. Each section is delimited by a banner comment; read on for what each one does.

## The training pipeline

The cell below runs end to end. Here is a map of its sections so you can navigate the code:

| Section | What it does |
|---|---|
| **Imports & deps** | Installs `onnx` / `onnxruntime` if missing, imports PyTorch & friends. |
| **Repro / device** | Fixes the random seed and selects GPU/CPU. |
| **Paths & config** | Hyperparameters: `IMG_SIZE=448`, `BATCH_SIZE=8`, `EPOCHS=20`, learning rates, EMA decay, teacher weight. |
| **Auto-detect dataset** | Finds the `*_image.png` files under `/kaggle/input`. |
| **Fold ImageNet norm** | Rewrites the first conv's weights so normalization is "free" at inference time. |
| **Teacher pre-computation** | Loads MiDaS once, predicts depth for all 9,200 images, caches it in FP16. Auto-detects and corrects depth direction via Pearson correlation. |
| **Dataset & augmentation** | `DepthDataset` with flips, rotation, crop, blur, gamma/brightness/contrast/saturation jitter, noise. |
| **Model — `DepthNet`** | MobileNetV2 encoder + depthwise-separable decoder + ReLU head. ~1.93 M params. |
| **EMA** | Maintains a shadow copy of weights; validation uses the EMA weights. |
| **Losses / metrics** | L1 + gradient + SSIM + scheduled scale-invariant loss, plus the teacher-alignment term and the RMSE / competition-score helpers. |
| **Training loop** | One-cycle LR, mixed precision (AMP), gradient clipping, early stopping on val RMSE. |
| **ONNX export** | Wraps the model to cast FP32↔FP16, exports to ONNX, patches IR version, verifies with ONNX Runtime. |
| **Benchmark** | Measures median inference time the same way the eval server does. |

Watch the `RMSE` column in the output drop from **0.28 → 0.11** over the 20 epochs.

In [37]:
"""
THIS IS THE ONE
AIGOAT 1.0 — Task 2: Monocular Depth Estimation (v4 — FP16 + EMA + Distill)
==============================================================================
Strategy: combine best of 1.42 solution + our MiDaS distillation
Key wins:
  1. FP16 ONNX export → ~5 MB (size score ≈ 2.25)
  2. EMA (decay=0.999) → smoother weights, better generalization
  3. Fold ImageNet norm into first conv → no explicit preprocess
  4. ReLU head → avoids sigmoid saturation
  5. Scheduled SI loss (ramp up)
  6. MiDaS teacher structural alignment
  7. 15% val split → reliable early stopping
  8. ORT benchmark → matches server
"""

# ============================================================================
# IMPORTS & DEPS
# ============================================================================
import os, sys, subprocess, math, random, time, gc
from pathlib import Path

import numpy as np

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import onnx
except Exception:
    _pip("onnx")
    import onnx

try:
    import onnxruntime as ort
except Exception:
    _pip("onnxruntime-gpu")
    import onnxruntime as ort

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from PIL import Image, ImageFilter
from tqdm.auto import tqdm

# ============================================================================
# REPRO / DEVICE
# ============================================================================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {device}")
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ============================================================================
# PATHS
# ============================================================================
OUTPUT_DIR = Path("/kaggle/working")
CKPT_DIR   = OUTPUT_DIR / "checkpoints"
ONNX_PATH  = OUTPUT_DIR / "depth_model.onnx"

IMG_SIZE       = 448
BATCH_SIZE     = 8
EPOCHS         = 20
LR             = 3e-4
ENC_LR_MUL     = 0.1
WEIGHT_DECAY   = 1e-4
USE_AMP        = True
VAL_RATIO      = 0.15
PATIENCE       = 15
TEACHER_WEIGHT = 0.35          # MiDaS structural alignment
EMA_DECAY      = 0.999

print(f"Config: img={IMG_SIZE} batch={BATCH_SIZE} epochs={EPOCHS} lr={LR}")
print(f"Val: {VAL_RATIO*100:.0f}% | EMA: {EMA_DECAY} | Teacher: {TEACHER_WEIGHT}")

# ============================================================================
# AUTO-DETECT DATASET
# ============================================================================
def find_dataset_dir():
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    counts = {}
    for p in root.rglob("*_image.png"):
        counts[p.parent] = counts.get(p.parent, 0) + 1
    if not counts:
        return None
    return max(counts.items(), key=lambda kv: kv[1])[0]

DATA_DIR = find_dataset_dir()
if DATA_DIR is None:
    raise RuntimeError("Could not find dataset with *_image.png under /kaggle/input")
print(f"✅ Dataset: {DATA_DIR}")
print(f"   examples: {[p.name for p in sorted(DATA_DIR.glob('*_image.png'))[:3]]}")

# ============================================================================
# FOLD IMAGENET NORM INTO FIRST CONV (no preprocess in forward!)
# ============================================================================
def fold_imagenet_norm_into_first_conv(module,
                                       mean=(0.485, 0.456, 0.406),
                                       std=(0.229, 0.224, 0.225)):
    first_conv = None
    for m in module.modules():
        if isinstance(m, nn.Conv2d):
            first_conv = m
            break
    if first_conv is None:
        raise RuntimeError("No Conv2d found")
    W = first_conv.weight.data
    dtype, dev = W.dtype, W.device
    mean_t = torch.tensor(mean, device=dev, dtype=dtype).view(1, -1, 1, 1)
    std_t  = torch.tensor(std,  device=dev, dtype=dtype).view(1, -1, 1, 1)
    if first_conv.bias is None:
        first_conv.bias = nn.Parameter(torch.zeros(W.shape[0], device=dev, dtype=dtype))
    W_new = W / std_t
    b_new = first_conv.bias.data - (W_new * mean_t).sum(dim=(1, 2, 3))
    first_conv.weight.data.copy_(W_new)
    first_conv.bias.data.copy_(b_new)

# ============================================================================
# TEACHER PRE-COMPUTATION (MiDaS small)
# ============================================================================
def precompute_teacher_depths(image_paths, depth_paths, batch_size=16):
    try:
        print("\n⏳ Loading MiDaS small teacher …")
        midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small", trust_repo=True)
        midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo=True)
        transform = midas_transforms.small_transform
        midas.eval().to(device)
        print("✅ MiDaS small loaded")
    except Exception as e:
        print(f"⚠️ Failed to load MiDaS: {e}")
        print("   Training will proceed WITHOUT distillation.")
        return None

    teacher_depths = []
    n = len(image_paths)
    print(f"⏳ Pre-computing teacher depths for {n} images …")

    for i in tqdm(range(0, n, batch_size), desc="Teacher inference"):
        batch_paths = image_paths[i:i+batch_size]
        batch_inputs = []
        for path in batch_paths:
            img = np.array(Image.open(path).convert("RGB"))
            batch_inputs.append(transform(img))
        batch = torch.cat(batch_inputs, dim=0).to(device)
        with torch.no_grad():
            with torch.amp.autocast("cuda", enabled=(USE_AMP and device == "cuda")):
                preds = midas(batch)
            preds = preds.float()
            preds = F.interpolate(preds.unsqueeze(1), size=(IMG_SIZE, IMG_SIZE),
                                  mode="bilinear", align_corners=False).squeeze(1)
        for j in range(preds.shape[0]):
            p = preds[j]
            if torch.isnan(p).any() or torch.isinf(p).any():
                p = torch.zeros(IMG_SIZE, IMG_SIZE)
            p = (p - p.min()) / (p.max() - p.min() + 1e-8)
            teacher_depths.append(p.cpu().half())

    del midas
    if device == "cuda": torch.cuda.empty_cache()
    gc.collect()

    result = torch.stack(teacher_depths)

    # Auto-detect depth direction
    corrs = []
    for k in range(min(20, n)):
        dp = depth_paths[k]
        if dp.exists():
            gt = np.array(Image.open(dp).convert("L"), dtype=np.float32).flatten()
            td = result[k].float().numpy().flatten()
            if gt.std() > 1e-6 and td.std() > 1e-6:
                corrs.append(np.corrcoef(gt, td)[0, 1])
    avg_corr = float(np.mean(corrs)) if corrs else 0.0
    print(f"   Pearson r = {avg_corr:.3f}")
    if avg_corr < -0.1:
        print("   ↻ Inverting teacher depths")
        result = 1.0 - result

    mem_gb = result.element_size() * result.nelement() / 1e9
    print(f"✅ Teacher: {result.shape} {result.dtype} ({mem_gb:.2f} GB)")
    return result

# ============================================================================
# DATASET
# ============================================================================
class DepthDataset(Dataset):
    def __init__(self, image_paths, depth_paths, teacher_depths=None, augment=True):
        self.images  = image_paths
        self.depths  = depth_paths
        self.teacher = teacher_depths
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

        depth_img = Image.open(self.depths[idx])
        depth_np = np.array(depth_img, dtype=np.float32)
        if depth_np.ndim == 3:
            depth_np = depth_np[:, :, 0]
        depth_pil = Image.fromarray(depth_np).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

        has_teacher = self.teacher is not None
        if has_teacher:
            t_np = self.teacher[idx].float().numpy()
            t_np = (t_np * 255).clip(0, 255).astype(np.uint8)
            teacher_pil = Image.fromarray(t_np)

        if self.augment:
            # Horizontal flip
            if random.random() < 0.5:
                img       = img.transpose(Image.FLIP_LEFT_RIGHT)
                depth_pil = depth_pil.transpose(Image.FLIP_LEFT_RIGHT)
                if has_teacher:
                    teacher_pil = teacher_pil.transpose(Image.FLIP_LEFT_RIGHT)

            # Light rotation ±5°
            if random.random() < 0.2:
                angle = random.uniform(-5, 5)
                img       = img.rotate(angle, resample=Image.BILINEAR,
                                       fillcolor=(128, 128, 128))
                depth_pil = depth_pil.rotate(angle, resample=Image.BILINEAR,
                                             fillcolor=0)
                if has_teacher:
                    teacher_pil = teacher_pil.rotate(angle, resample=Image.BILINEAR,
                                                     fillcolor=0)

            # Random crop & resize
            if random.random() < 0.2:
                scale = random.uniform(0.88, 1.0)
                cs = int(IMG_SIZE * scale)
                l = random.randint(0, IMG_SIZE - cs)
                t = random.randint(0, IMG_SIZE - cs)
                box = (l, t, l + cs, t + cs)
                img       = img.crop(box).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
                depth_pil = depth_pil.crop(box).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
                if has_teacher:
                    teacher_pil = teacher_pil.crop(box).resize((IMG_SIZE, IMG_SIZE),
                                                                Image.BILINEAR)

            # Gaussian blur
            if random.random() < 0.12:
                img = img.filter(ImageFilter.GaussianBlur(
                    radius=random.uniform(0.1, 0.8)))

        # Convert to numpy
        img_np = np.array(img, dtype=np.float32) / 255.0
        depth_np = np.array(depth_pil, dtype=np.float32)
        if depth_np.max() > 1.0:
            depth_np = depth_np / 255.0
        depth_np = np.clip(depth_np, 0.0, 1.0)

        if self.augment:
            # Gamma
            if random.random() < 0.25:
                img_np = np.clip(img_np ** random.uniform(0.85, 1.15), 0, 1)
            # Brightness
            if random.random() < 0.3:
                img_np = np.clip(img_np * random.uniform(0.8, 1.2), 0, 1)
            # Contrast
            if random.random() < 0.3:
                mu = img_np.mean()
                img_np = np.clip((img_np - mu) * random.uniform(0.8, 1.2) + mu, 0, 1)
            # Saturation
            if random.random() < 0.2:
                gray = img_np.mean(axis=2, keepdims=True)
                alpha = random.uniform(0.8, 1.2)
                img_np = np.clip(alpha * img_np + (1 - alpha) * gray, 0, 1)
            # Noise
            if random.random() < 0.15:
                img_np = np.clip(img_np + np.random.normal(0, 0.015, img_np.shape).astype(np.float32), 0, 1)
            # Channel shuffle
            if random.random() < 0.08:
                img_np = img_np[:, :, np.random.permutation(3)]

        img_t   = torch.from_numpy(np.transpose(img_np, (2, 0, 1)).copy())
        depth_t = torch.from_numpy(depth_np.copy())

        if has_teacher:
            teacher_np = np.array(teacher_pil, dtype=np.float32) / 255.0
            teacher_t  = torch.from_numpy(teacher_np.copy())
        else:
            teacher_t = torch.zeros(IMG_SIZE, IMG_SIZE)

        return img_t, depth_t, teacher_t


def build_dataloaders():
    image_paths = sorted(DATA_DIR.glob("*_image.png"))
    depth_paths = [Path(str(p).replace("_image.png", "_depth_map.png")) for p in image_paths]
    valid = [(i, d) for i, d in zip(image_paths, depth_paths) if d.exists()]
    image_paths = [v[0] for v in valid]
    depth_paths = [v[1] for v in valid]
    n = len(image_paths)
    if n < 10:
        raise RuntimeError(f"Too few valid pairs: {n}")
    print(f"\nFound {n} valid image–depth pairs")

    teacher_depths = precompute_teacher_depths(image_paths, depth_paths)
    use_teacher = teacher_depths is not None

    idxs = list(range(n))
    random.Random(SEED).shuffle(idxs)
    n_val   = max(1, int(VAL_RATIO * n))
    n_train = n - n_val
    train_idx = idxs[n_val:]     # train = everything after val
    val_idx   = idxs[:n_val]     # val = first n_val after shuffle

    train_teacher = None
    if use_teacher:
        # Re-index teacher for training subset only
        train_teacher = teacher_depths[train_idx]
        del teacher_depths
        gc.collect()

    train_ds = DepthDataset(
        [image_paths[i] for i in train_idx],
        [depth_paths[i] for i in train_idx],
        teacher_depths=train_teacher, augment=True,
    )
    val_ds = DepthDataset(
        [image_paths[i] for i in val_idx],
        [depth_paths[i] for i in val_idx],
        teacher_depths=None, augment=False,
    )
    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds, BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)
    print(f"Train: {n_train} | Val: {n_val}")
    print(f"Teacher: {'✅ ENABLED' if use_teacher else '❌ DISABLED'}")
    return train_loader, val_loader, use_teacher

# ============================================================================
# MODEL — MobileNetV2 + DepthwiseSep Decoder + ReLU head + folded norm (<5MB)
# ============================================================================
class UpBlock(nn.Module):
    """Depthwise separable decoder block — MobileNet-style, far fewer params."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        total_in = in_ch + skip_ch
        self.conv = nn.Sequential(
            # Depthwise separable conv 1
            nn.Conv2d(total_in, total_in, 3, 1, 1, groups=total_in, bias=False),
            nn.BatchNorm2d(total_in), nn.ReLU6(inplace=True),
            nn.Conv2d(total_in, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU6(inplace=True),
            # Depthwise separable conv 2
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, groups=out_ch, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU6(inplace=True),
            nn.Conv2d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU6(inplace=True),
        )

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class DepthNet(nn.Module):
    """
    MobileNetV2 encoder + depthwise-separable decoder (<5 MB FP16).
    ImageNet norm folded into first conv — input is raw [0,1] RGB.
    ReLU head (no sigmoid) — avoids gradient saturation.
    """
    def __init__(self):
        super().__init__()
        mob = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        f = mob.features
        fold_imagenet_norm_into_first_conv(f)

        self.enc0 = f[0:2]      # 16 ch  /2
        self.enc1 = f[2:4]      # 24 ch  /4
        self.enc2 = f[4:7]      # 32 ch  /8
        self.enc3 = f[7:14]     # 96 ch  /16
        self.enc4 = f[14:18]    # 320 ch /32

        self.bridge = nn.Sequential(
            nn.Conv2d(320, 192, 1, bias=False),
            nn.BatchNorm2d(192), nn.ReLU6(inplace=True),
            nn.Dropout2d(0.10),
        )

        self.up4 = UpBlock(192, 96,  96)
        self.up3 = UpBlock( 96, 32,  48)
        self.up2 = UpBlock( 48, 24,  32)
        self.up1 = UpBlock( 32, 16,  24)

        # ReLU head (depthwise + pointwise, not sigmoid!)
        self.head = nn.Sequential(
            nn.Conv2d(24, 24, 3, 1, 1, groups=24, bias=False),
            nn.BatchNorm2d(24), nn.ReLU6(inplace=True),
            nn.Conv2d(24, 1, 1),
            nn.ReLU(inplace=False),
        )

    def forward(self, x):
        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        d  = self.bridge(e4)
        d  = self.up4(d, e3)
        d  = self.up3(d, e2)
        d  = self.up2(d, e1)
        d  = self.up1(d, e0)
        d  = F.interpolate(d, size=(x.shape[2], x.shape[3]),
                           mode="bilinear", align_corners=False)
        d  = self.head(d)
        return d.squeeze(1)

_m = DepthNet()
_total = sum(p.numel() for p in _m.parameters())
print(f"DepthNet(MobileNetV2) — Total: {_total:,} params")
print(f"  FP32 ONNX ≈ {_total*4/1e6:.1f} MB  |  FP16 ONNX ≈ {_total*2/1e6:.1f} MB")
del _m

# ============================================================================
# EMA
# ============================================================================
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.detach().clone()
                       for n, p in model.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if n in self.shadow:
                self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def apply_to(self, model):
        self.backup = {}
        for n, p in model.named_parameters():
            if n in self.shadow:
                self.backup[n] = p.detach().clone()
                p.copy_(self.shadow[n])

    @torch.no_grad()
    def restore(self, model):
        for n, p in model.named_parameters():
            if n in getattr(self, "backup", {}):
                p.copy_(self.backup[n])
        self.backup = {}

# ============================================================================
# LOSSES / METRICS
# ============================================================================
def normalize_depth(d, eps=1e-8):
    B = d.shape[0]
    flat = d.reshape(B, -1)
    lo = flat.min(dim=1)[0].view(B, 1, 1)
    hi = flat.max(dim=1)[0].view(B, 1, 1)
    return (d - lo) / (hi - lo + eps)

def gradient_loss(pred, gt):
    dx_p = pred[:, :, 1:] - pred[:, :, :-1]
    dy_p = pred[:, 1:, :] - pred[:, :-1, :]
    dx_g = gt[:, :, 1:]   - gt[:, :, :-1]
    dy_g = gt[:, 1:, :]   - gt[:, :-1, :]
    return F.l1_loss(dx_p, dx_g) + F.l1_loss(dy_p, dy_g)

def scale_invariant_loss(pred, gt, lam=0.5, eps=1e-6):
    diff = torch.log(pred.clamp(min=eps)) - torch.log(gt.clamp(min=eps))
    return torch.sqrt((diff ** 2).mean() - lam * (diff.mean()) ** 2 + eps)

def ssim_loss_fn(pred, gt, win=7):
    C1, C2, pad = 0.01**2, 0.03**2, win // 2
    p4 = pred.unsqueeze(1); g4 = gt.unsqueeze(1)
    mu_p = F.avg_pool2d(p4, win, 1, pad)
    mu_g = F.avg_pool2d(g4, win, 1, pad)
    s_pp = (F.avg_pool2d(p4*p4, win, 1, pad) - mu_p**2).clamp(min=0)
    s_gg = (F.avg_pool2d(g4*g4, win, 1, pad) - mu_g**2).clamp(min=0)
    s_pg = F.avg_pool2d(p4*g4, win, 1, pad) - mu_p*mu_g
    num  = (2*mu_p*mu_g + C1) * (2*s_pg + C2)
    den  = (mu_p**2 + mu_g**2 + C1) * (s_pp + s_gg + C2) + 1e-8
    return 1.0 - (num / den).mean()


class DepthLoss(nn.Module):
    """L1 + Grad + SSIM + scheduled SI  +  optional teacher alignment."""
    def __init__(self, teacher_weight=0.0):
        super().__init__()
        self.tw = teacher_weight

    def forward(self, pred, gt, teacher=None, si_w=0.0):
        pn = normalize_depth(pred)
        gn = normalize_depth(gt)

        l1   = F.l1_loss(pn, gn)
        grad = gradient_loss(pn, gn)
        ssim = ssim_loss_fn(pn, gn)
        si   = scale_invariant_loss(pn, gn) if si_w > 0 else pn.new_tensor(0.0)
        loss = l1 + 0.5*grad + 0.2*ssim + si_w*si

        align_val = 0.0
        if teacher is not None and self.tw > 0:
            tn = normalize_depth(teacher)
            # Structural alignment: L1 + gradient matching with teacher
            t_l1   = F.l1_loss(pn, tn)
            t_grad = gradient_loss(pn, tn)
            align  = t_l1 + 0.3 * t_grad
            loss   = loss + self.tw * align
            align_val = align.item()

        return loss, l1.item(), grad.item(), ssim.item(), si.item(), align_val


@torch.no_grad()
def calc_rmse(pred, gt, eps=1e-8):
    pn = normalize_depth(pred)
    gn = normalize_depth(gt)
    return torch.sqrt(F.mse_loss(pn, gn) + eps).item()

def estimate_competition_score(rmse, size_mb, t_sec):
    acc   = 4.0 / (4.0 + (10.0 * rmse) ** 2)
    size  = (50.0 - size_mb) / 20.0
    speed = 0.16 + math.log10(max(7.0 - (2 / 5.33) * t_sec, 0.01))
    total = acc * size * speed
    print(f"  Accuracy  = {acc:.4f}  (RMSE {rmse:.4f})")
    print(f"  Size      = {size:.4f}  ({size_mb:.2f} MB)")
    print(f"  Speed     = {speed:.4f}  ({t_sec:.3f} s)")
    print(f"  ── Score  = {total:.4f}")
    return total

# ============================================================================
# TRAINING
# ============================================================================
def train_model():
    train_loader, val_loader, use_teacher = build_dataloaders()

    model     = DepthNet().to(device)
    criterion = DepthLoss(teacher_weight=TEACHER_WEIGHT if use_teacher else 0.0)
    ema       = EMA(model, decay=EMA_DECAY)

    enc_params = [p for n, p in model.named_parameters() if "enc" in n]
    dec_params = [p for n, p in model.named_parameters() if "enc" not in n]
    optimizer  = torch.optim.AdamW([
        {"params": enc_params, "lr": LR * ENC_LR_MUL},
        {"params": dec_params, "lr": LR},
    ], weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[LR * ENC_LR_MUL, LR],
        epochs=EPOCHS, steps_per_epoch=len(train_loader),
        pct_start=0.05, anneal_strategy="cos",
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device == "cuda"))

    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    best_rmse    = float("inf")
    patience_ctr = 0

    tag = "MobileNetV2 + FP16 + EMA"
    if use_teacher:
        tag += " + MiDaS"
    print(f"\n{'='*70}")
    print(f"Training — {EPOCHS} epochs, {tag}")
    print(f"Loss = L1 + 0.5·Grad + 0.2·SSIM + scheduled(SI)")
    if use_teacher:
        print(f"  + {TEACHER_WEIGHT}·(L1_teacher + 0.3·Grad_teacher)")
    print(f"Val: {VAL_RATIO*100:.0f}% | Patience: {PATIENCE} | EMA: {EMA_DECAY}")
    print(f"{'='*70}\n")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0

        # Scheduled SI loss: ramp up from 0 to 0.35 over epochs 4-10
        si_w = 0.0 if epoch <= 3 else min(0.35, 0.35 * (epoch - 3) / 7.0)

        for imgs, depths, teacher_d in tqdm(train_loader,
                                            desc=f"Epoch {epoch}/{EPOCHS}",
                                            leave=False):
            imgs   = imgs.to(device, non_blocking=True)
            depths = depths.to(device, non_blocking=True)
            td     = teacher_d.to(device, non_blocking=True) if use_teacher else None

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(USE_AMP and device == "cuda")):
                pred = model(imgs)
                loss, *_ = criterion(pred, depths, td, si_w=si_w)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            scheduler.step()
            ema.update(model)
            running_loss += loss.item()

        avg_loss = running_loss / len(train_loader)

        # ── Validate with EMA weights ──
        model.eval()
        ema.apply_to(model)
        v_rmse = 0.0
        with torch.no_grad():
            for imgs, depths, _ in val_loader:
                imgs   = imgs.to(device)
                depths = depths.to(device)
                with torch.amp.autocast("cuda", enabled=(USE_AMP and device == "cuda")):
                    pred = model(imgs)
                v_rmse += calc_rmse(pred.float(), depths)
        v_rmse /= max(1, len(val_loader))
        lr_now = optimizer.param_groups[1]["lr"]

        print(f"Epoch {epoch:3d} | loss={avg_loss:.4f} lr={lr_now:.2e} si_w={si_w:.2f} | "
              f"RMSE={v_rmse:.4f}")

        if v_rmse < best_rmse:
            best_rmse = v_rmse
            patience_ctr = 0
            torch.save(model.state_dict(), CKPT_DIR / "best.pth")
            print(f"         ★ New best RMSE = {best_rmse:.4f}")
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f"  ⏹ Early stopping (no improvement for {PATIENCE} epochs)")
                ema.restore(model)
                break

        ema.restore(model)

    print(f"\n✓ Best validation RMSE: {best_rmse:.4f}")
    model.load_state_dict(torch.load(CKPT_DIR / "best.pth", map_location=device))
    return model, best_rmse

# ============================================================================
# ONNX EXPORT  (FP16 weights, FP32 I/O → ~5 MB)
# ============================================================================
def export_onnx(model):
    class FP16ExportWrapper(nn.Module):
        def __init__(self, fp16_model):
            super().__init__()
            self.m = fp16_model
        def forward(self, x):
            x = x.half()
            y = self.m(x)
            return y.float()

    model.eval()
    model_fp16 = model.to(device).half().eval()
    wrapped    = FP16ExportWrapper(model_fp16).to(device).eval()
    dummy      = torch.randn(8, 3, IMG_SIZE, IMG_SIZE, device=device, dtype=torch.float32)

    try:
        torch.onnx.export(
            wrapped, dummy, str(ONNX_PATH),
            input_names=["input"], output_names=["output"],
            opset_version=14, do_constant_folding=True,
            dynamic_axes=None,
            training=torch.onnx.TrainingMode.EVAL,
            dynamo=False,
        )
    except TypeError:
        torch.onnx.export(
            wrapped, dummy, str(ONNX_PATH),
            input_names=["input"], output_names=["output"],
            opset_version=14, do_constant_folding=True,
            dynamic_axes=None,
            training=torch.onnx.TrainingMode.EVAL,
        )

    # Patch IR version for server compatibility
    m = onnx.load(str(ONNX_PATH))
    if m.ir_version > 9:
        m.ir_version = 9
        onnx.save(m, str(ONNX_PATH))

    size_mb = os.path.getsize(ONNX_PATH) / (1024 ** 2)
    print(f"\n✅ ONNX exported → {ONNX_PATH}")
    print(f"   Size: {size_mb:.2f} MB  |  IR version: {onnx.load(str(ONNX_PATH)).ir_version}")

    # ORT verification
    sess = ort.InferenceSession(str(ONNX_PATH),
                                providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    inp = np.random.rand(8, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
    out = sess.run(None, {"input": inp})[0]
    print(f"   ORT: {inp.shape} → {out.shape}")
    print(f"   range: [{out.min():.4f}, {out.max():.4f}] | nan={np.isnan(out).any()}")
    assert out.shape == (8, IMG_SIZE, IMG_SIZE), f"Bad shape: {out.shape}"
    assert not np.isnan(out).any() and not np.isinf(out).any()
    print("   ✅ ONNX verification passed!")
    return size_mb

# ============================================================================
# BENCHMARK (ORT — matches server measurement)
# ============================================================================
def benchmark_speed_ort():
    sess = ort.InferenceSession(str(ONNX_PATH),
                                providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    inp = np.random.rand(8, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
    # warmup
    for _ in range(3):
        _ = sess.run(None, {"input": inp})
    times = []
    for _ in range(5):
        t0 = time.time()
        _ = sess.run(None, {"input": inp})
        times.append(time.time() - t0)
    med = float(np.median(times))
    print(f"ORT median inference (batch=8): {med:.4f} s")
    print(f"ORT all times: {[f'{t:.4f}' for t in times]}")
    return med

# ============================================================================
# RUN
# ============================================================================
model, best_rmse = train_model()
size_mb  = export_onnx(model)
median_s = benchmark_speed_ort()

print(f"\n{'='*60}")
print("Estimated competition score:")
estimate_competition_score(best_rmse, size_mb, median_s)
print(f"{'='*60}")
print("\n✅ Done! Submit depth_model.onnx via the next cell.")

PyTorch 2.8.0+cu126 | Device: cuda
GPU: Tesla T4
Memory: 15.6 GB
Config: img=448 batch=8 epochs=20 lr=0.0003
Val: 15% | EMA: 0.999 | Teacher: 0.35
✅ Dataset: /kaggle/input/ai-goat-1-0-task2-dataset/train
   examples: ['000001_image.png', '000003_image.png', '000004_image.png']
DepthNet(MobileNetV2) — Total: 1,933,121 params
  FP32 ONNX ≈ 7.7 MB  |  FP16 ONNX ≈ 3.9 MB

Found 9200 valid image–depth pairs

⏳ Loading MiDaS small teacher …


Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master
Using cache found in /root/.cache/torch/hub/rwightman_gen-efficientnet-pytorch_master


Loading weights:  None
✅ MiDaS small loaded
⏳ Pre-computing teacher depths for 9200 images …


Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


Teacher inference:   0%|          | 0/575 [00:00<?, ?it/s]

...

Epoch  19 | loss=0.4976 lr=2.04e-06 si_w=0.35 | RMSE=0.1084


Epoch 20/20:   0%|          | 0/977 [00:00<?, ?it/s]

Epoch  20 | loss=0.4987 lr=1.20e-09 si_w=0.35 | RMSE=0.1079

✓ Best validation RMSE: 0.1077


/tmp/ipykernel_55/1685945658.py:662: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(



✅ ONNX exported → /kaggle/working/depth_model.onnx
   Size: 3.71 MB  |  IR version: 7
   ORT: (8, 3, 448, 448) → (8, 448, 448)
   range: [0.0000, 2.1895] | nan=False
   ✅ ONNX verification passed!
ORT median inference (batch=8): 0.0480 s
ORT all times: ['0.0482', '0.0480', '0.0481', '0.0400', '0.0352']

Estimated competition score:
  Accuracy  = 0.7752  (RMSE 0.1077)
  Size      = 2.3147  (3.71 MB)
  Speed     = 1.0040  (0.048 s)
  ── Score  = 1.8015

✅ Done! Submit depth_model.onnx via the next cell.


## Submitting to the leaderboard

The exported `depth_model.onnx` is uploaded to the competition's evaluation server using the organizers' `GOAT` toolkit. The server runs our ONNX model on the held-out test set (we never see those ground-truth depths), times the forward pass, and computes the composite score.

The output below is the **actual submission result**: the model scored **1.47** for a final rank of **#3** on the leaderboard, with a measured inference time of 1.37 s and a 3.71 MB model.

> The `team_ps` (team password) is the original hackathon credential and is left here only as a record of the run — it is tied to a finished competition and grants no further access.

In [ ]:
# =========================
# SUBMIT TO LEADERBOARD
# =========================
!git clone --depth 1 https://github.com/chater-marzougui/MLS-GOAT.git
!mv MLS-GOAT/GOAT .
!rm -rf MLS-GOAT

from GOAT import submit, leaderboard

team_name = "PPP: PeniParkersPrime"
team_ps = "<redacted>"   # original hackathon credential removed before publishing

# Submit ONNX model for Task 2
submit.challenge2(str(ONNX_PATH), team_name, team_ps)
leaderboard.getLB_chall2(team_name)